Questo notebook fa parte di **Pensare in Python**, traduzione italiana non ufficiale di [*Think Python*, 3ª edizione](https://allendowney.github.io/ThinkPython/) di Allen B. Downey. Puoi leggere tutto il libro tradotto su <https://pensare-in-python.github.io/3a-edizione/>.

Le versioni stampata ed ebook dell'originale si possono acquistare da [Bookshop.org](https://bookshop.org/a/98697/9781098155438) e [Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325) (link di affiliazione dell'autore).

Se trovi utile questo libro, valuta di [sostenere i materiali didattici liberi e aperti](https://greenteapress.com/support/?book=tp3) di Green Tea Press.

In [ ]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

# Analisi e generazione di testo

[Esegui questo notebook su Colab](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap12.ipynb)

A questo punto abbiamo visto le strutture dati fondamentali di Python – liste, dizionari e tuple – e alcuni algoritmi che le usano.
In questo capitolo le useremo per esplorare l'analisi del testo e la generazione di Markov:

* L'analisi del testo è un modo per descrivere le relazioni statistiche tra le parole di un documento, come la probabilità che una parola sia seguita da un'altra, e

* La generazione di Markov è un modo per generare nuovo testo con parole e frasi simili a quelle del testo originale.

Questi algoritmi sono simili ad alcune parti di un modello linguistico di grandi dimensioni (LLM), che è il componente chiave di un chatbot.

Inizieremo contando il numero di volte in cui ogni parola compare in un libro.
Poi esamineremo le coppie di parole e creeremo una lista delle parole che possono seguire ciascuna parola.
Realizzeremo una versione semplice di un generatore di Markov e, come esercizio, avrai l'occasione di realizzarne una versione più generale.

## Parole distinte

Come primo passo verso l'analisi del testo, leggiamo un libro – *Lo strano caso del dottor Jekyll e del signor Hyde* di Robert Louis Stevenson – e contiamo il numero di parole distinte.
Le istruzioni per scaricare il libro si trovano nel notebook di questo capitolo.

La cella seguente scarica il libro da Project Gutenberg.

In [ ]:
download('https://github.com/AllenDowney/ThinkPython/raw/v3/pg43.txt');

La versione disponibile su Project Gutenberg include informazioni sul libro all'inizio e informazioni sulla licenza alla fine.
Useremo `clean_file` del capitolo 8 per rimuovere questo materiale e scrivere un file "pulito" che contiene solo il testo del libro.

In [ ]:
def is_special_line(line):
    return line.strip().startswith('*** ')

In [ ]:
def clean_file(input_file, output_file):
    reader = open(input_file, encoding='utf-8')
    writer = open(output_file, 'w', encoding='utf-8')

    for line in reader:
        if is_special_line(line):
            break

    for line in reader:
        if is_special_line(line):
            break
        writer.write(line)
        
    reader.close()
    writer.close()

In [ ]:
filename = 'dr_jekyll.txt'

In [ ]:
clean_file('pg43.txt', filename)

Useremo un ciclo `for` per leggere le righe del file e `split` per dividere le righe in parole.
Poi, per tenere traccia delle parole distinte, memorizzeremo ogni parola come chiave in un dizionario.

L'argomento keyword `encoding='utf-8'` non compare nel libro stampato.
Serve solo su Windows.

In [ ]:
unique_words = {}
for line in open(filename, encoding='utf-8'):
    seq = line.split()
    for word in seq:
        unique_words[word] = 1

len(unique_words)

La lunghezza del dizionario è il numero di parole distinte – circa `6000` con questo modo di contare.
Ma se le esaminiamo, vedremo che alcune non sono parole valide.

Per esempio, guardiamo le parole più lunghe in `unique_words`.
Possiamo usare `sorted` per ordinare le parole, passando la funzione `len` come argomento keyword così che le parole vengano ordinate per lunghezza.

In [ ]:
sorted(unique_words, key=len)[-5:]

L'indice di slice, `[-5:]`, seleziona gli ultimi `5` elementi della lista ordinata, che sono le parole più lunghe.

La lista include alcune parole legittimamente lunghe, come "circumscription", e alcune parole con trattino, come "chocolate-coloured".
Ma alcune delle "parole" più lunghe sono in realtà due parole separate da un trattino lungo.
E altre parole includono punteggiatura come punti, punti esclamativi e virgolette.

Quindi, prima di andare avanti, occupiamoci dei trattini lunghi e del resto della punteggiatura.

## Punteggiatura

Per identificare le parole nel testo, dobbiamo affrontare due problemi:

* Quando in una riga compare un trattino lungo (*dash*), dovremmo sostituirlo con uno spazio – così, quando usiamo `split`, le parole saranno separate.

* Dopo aver diviso le parole, possiamo usare `strip` per rimuovere la punteggiatura.

Per gestire il primo problema, possiamo usare la funzione seguente, che riceve una stringa, sostituisce i trattini lunghi con spazi, divide la stringa e restituisce la lista risultante.

In [ ]:
def split_line(line):
    return line.replace('—', ' ').split()

Nota che `split_line` sostituisce solo i trattini lunghi, non i trattini corti (*hyphen*) delle parole composte.
Ecco un esempio.

In [ ]:
split_line('coolness—frightened')

Ora, per rimuovere la punteggiatura dall'inizio e dalla fine di ogni parola, possiamo usare `strip`, ma ci serve un elenco dei caratteri considerati punteggiatura.

I caratteri delle stringhe Python sono in Unicode, uno standard internazionale usato per rappresentare le lettere di quasi tutti gli alfabeti, i numeri, i simboli, i segni di punteggiatura e altro.
Il modulo `unicodedata` fornisce una funzione `category` che possiamo usare per capire quali caratteri sono punteggiatura.
Data una lettera, restituisce una stringa con informazioni sulla categoria a cui appartiene.

In [ ]:
import unicodedata

unicodedata.category('A')

La stringa di categoria di `'A'` è `'Lu'` – la `'L'` significa che è una lettera e la `'u'` che è maiuscola (*uppercase*).

La stringa di categoria di `'.'` è `'Po'` – la `'P'` significa che è punteggiatura e la `'o'` che la sua sottocategoria è "altro" (*other*).

In [ ]:
unicodedata.category('.')

Possiamo trovare i segni di punteggiatura del libro cercando i caratteri con categorie che iniziano con `'P'`.
Il ciclo seguente memorizza in un dizionario i segni di punteggiatura distinti.

In [ ]:
punc_marks = {}
for line in open(filename, encoding='utf-8'):
    for char in line:
        category = unicodedata.category(char)
        if category.startswith('P'):
            punc_marks[char] = 1

Per creare un elenco di segni di punteggiatura, possiamo unire le chiavi del dizionario in una stringa.

In [ ]:
punctuation = ''.join(punc_marks)
print(punctuation)

Ora che sappiamo quali caratteri del libro sono punteggiatura, possiamo scrivere una funzione che riceve una parola, toglie la punteggiatura dall'inizio e dalla fine e la converte in minuscolo.

In [ ]:
def clean_word(word):
    return word.strip(punctuation).lower()

Ecco un esempio.

In [ ]:
clean_word('“Behold!”')

Poiché `strip` rimuove caratteri dall'inizio e dalla fine, lascia intatte le parole con trattino.

In [ ]:
clean_word('pocket-handkerchief')

Ecco ora un ciclo che usa `split_line` e `clean_word` per identificare le parole distinte del libro.

In [ ]:
unique_words2 = {}
for line in open(filename, encoding='utf-8'):
    for word in split_line(line):
        word = clean_word(word)
        unique_words2[word] = 1

len(unique_words2)

Con questa definizione più rigorosa di parola, ci sono circa 4000 parole distinte.
E possiamo confermare che l'elenco delle parole più lunghe è stato ripulito.

In [ ]:
sorted(unique_words2, key=len)[-5:]

Ora vediamo quante volte viene usata ogni parola.

## Frequenze delle parole

Il ciclo seguente calcola la frequenza di ogni parola distinta.

In [ ]:
word_counter = {}
for line in open(filename, encoding='utf-8'):
    for word in split_line(line):
        word = clean_word(word)
        if word not in word_counter:
            word_counter[word] = 1
        else:
            word_counter[word] += 1

La prima volta che vediamo una parola, inizializziamo la sua frequenza a `1`. Se in seguito vediamo di nuovo la stessa parola, incrementiamo la sua frequenza.

Per vedere quali parole compaiono più spesso, possiamo usare `items` per ottenere le coppie chiave-valore di `word_counter` e ordinarle in base al secondo elemento della coppia, che è la frequenza.
Prima definiremo una funzione che seleziona il secondo elemento.

In [ ]:
def second_element(t):
    return t[1]

Ora possiamo usare `sorted` con due argomenti keyword:

* `key=second_element` significa che gli elementi verranno ordinati in base alle frequenze delle parole.

* `reverse=True` significa che gli elementi verranno ordinati in ordine inverso, con le parole più frequenti per prime.

In [ ]:
items = sorted(word_counter.items(), key=second_element, reverse=True)

Ecco le cinque parole più frequenti.

In [ ]:
for word, freq in items[:5]:
    print(freq, word, sep='\t')

Nella prossima sezione incapsuleremo questo ciclo in una funzione.
E la useremo per illustrare una nuova funzionalità – i parametri facoltativi.

## Parametri facoltativi

Abbiamo usato funzioni predefinite che ricevono parametri facoltativi.
Per esempio, `round` riceve un parametro facoltativo chiamato `ndigits` che indica quante cifre decimali mantenere.

In [ ]:
round(3.141592653589793, ndigits=3)

Ma non si tratta solo di funzioni predefinite – anche noi possiamo scrivere funzioni con parametri facoltativi.
Per esempio, la funzione seguente riceve due parametri, `word_counter` e `num`.

In [ ]:
def print_most_common(word_counter, num=5):
    items = sorted(word_counter.items(), key=second_element, reverse=True)

    for word, freq in items[:num]:
        print(freq, word, sep='\t')

Il secondo parametro sembra un'istruzione di assegnazione, ma non lo è – è un parametro facoltativo.

Se chiami questa funzione con un solo argomento, `num` riceve il **valore predefinito** (*default value*), che è `5`.

In [ ]:
print_most_common(word_counter)

Se chiami questa funzione con due argomenti, il secondo argomento viene assegnato a `num` al posto del valore predefinito.

In [ ]:
print_most_common(word_counter, 3)

In questo caso diremmo che l'argomento facoltativo **sovrascrive** (*overrides*) il valore predefinito.

Se una funzione ha sia parametri obbligatori sia facoltativi, tutti i parametri obbligatori devono venire prima, seguiti da quelli facoltativi.

In [ ]:
%%expect SyntaxError

def bad_function(n=5, word_counter):
    return None

## Sottrazione di dizionari

Supponiamo di voler fare il controllo ortografico di un libro – cioè trovare un elenco di parole che potrebbero essere scritte male.
Un modo per farlo è trovare le parole del libro che non compaiono in un elenco di parole valide.
Nei capitoli precedenti abbiamo usato un elenco di parole considerate valide in giochi di parole come lo Scarabeo (Scrabble).
Ora useremo questo elenco per controllare l'ortografia di Robert Louis Stevenson.

Possiamo pensare a questo problema come a una sottrazione di insiemi – cioè vogliamo trovare tutte le parole di un insieme (le parole del libro) che non sono nell'altro (le parole dell'elenco).

La cella seguente scarica l'elenco di parole.

In [ ]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

Come abbiamo fatto prima, possiamo leggere il contenuto di `words.txt` e dividerlo in una lista di stringhe.

In [ ]:
word_list = open('words.txt').read().split()

Poi memorizzeremo le parole come chiavi di un dizionario, così possiamo usare l'operatore `in` per verificare rapidamente se una parola è valida.

In [ ]:
valid_words = {}
for word in word_list:
    valid_words[word] = 1

Ora, per identificare le parole che compaiono nel libro ma non nell'elenco, useremo `subtract`, che riceve due dizionari come parametri e restituisce un nuovo dizionario che contiene tutte le chiavi del primo che non sono nel secondo.

In [ ]:
def subtract(d1, d2):
    res = {}
    for key in d1:
        if key not in d2:
            res[key] = d1[key]
    return res

Ecco come la usiamo.

In [ ]:
diff = subtract(word_counter, valid_words)

Per ottenere un campione di parole che potrebbero essere scritte male, possiamo stampare le parole più comuni di `diff`.

In [ ]:
print_most_common(diff)

Le parole "scritte male" più comuni sono soprattutto nomi e alcune parole di una sola lettera (il signor Utterson è l'amico e avvocato del dottor Jekyll).

Se selezioniamo le parole che compaiono una sola volta, è più probabile che siano veri errori di ortografia.
Possiamo farlo scorrendo gli elementi e creando una lista delle parole con frequenza `1`.

In [ ]:
singletons = []
for word, freq in diff.items():
    if freq == 1:
        singletons.append(word)

Ecco gli ultimi elementi della lista.

In [ ]:
singletons[-5:]

La maggior parte sono parole valide che non sono nell'elenco.
Ma `'reindue'` sembra un errore di ortografia per `'reinduce'`, quindi almeno abbiamo trovato un errore vero.

## Numeri casuali

Come passo verso la generazione di testo di Markov, sceglieremo ora una sequenza casuale di parole da `word_counter`.
Ma prima parliamo di casualità.

A parità di input, la maggior parte dei programmi per computer è **deterministica** (*deterministic*), il che significa che genera ogni volta lo stesso output.
Il determinismo di solito è una buona cosa, perché ci aspettiamo che lo stesso calcolo dia lo stesso risultato.
Per alcune applicazioni, però, vogliamo che il computer sia imprevedibile.
I giochi sono un esempio, ma ce ne sono altri.

Rendere un programma davvero non deterministico si rivela difficile, ma ci sono modi per simularlo.
Uno è usare algoritmi che generano numeri **pseudocasuali** (*pseudorandom*).
I numeri pseudocasuali non sono davvero casuali perché sono generati da un calcolo deterministico, ma guardando solo i numeri è quasi impossibile distinguerli da quelli casuali.

Il modulo `random` fornisce funzioni che generano numeri pseudocasuali – che d'ora in poi chiamerò semplicemente "casuali".
Possiamo importarlo così.

In [ ]:
import random

In [ ]:
# this cell initializes the random number generator so it 
# generates the same sequence each time the notebook runs.

random.seed(4)

Il modulo `random` fornisce una funzione chiamata `choice` che sceglie a caso un elemento da una lista, con ogni elemento che ha la stessa probabilità di essere scelto.

In [ ]:
t = [1, 2, 3]
random.choice(t)

Se chiami di nuovo la funzione, potresti ottenere di nuovo lo stesso elemento, o uno diverso.

In [ ]:
random.choice(t)

Nel lungo periodo, ci aspettiamo di ottenere ogni elemento all'incirca lo stesso numero di volte.

Se usi `choice` con un dizionario, ottieni un `KeyError`.

In [ ]:
%%expect KeyError

random.choice(word_counter)

Per scegliere una chiave a caso, devi mettere le chiavi in una lista e poi chiamare `choice`.

In [ ]:
words = list(word_counter)
random.choice(words)

Se generiamo una sequenza casuale di parole, non ha molto senso.

In [ ]:
for i in range(6):
    word = random.choice(words)
    print(word, end=' ')

Parte del problema è che non teniamo conto del fatto che alcune parole sono più comuni di altre.
I risultati saranno migliori se scegliamo le parole con "pesi" diversi, così che alcune vengano scelte più spesso di altre.

Se usiamo i valori di `word_counter` come pesi, ogni parola viene scelta con una probabilità che dipende dalla sua frequenza.

In [ ]:
weights = word_counter.values()

Il modulo `random` fornisce un'altra funzione chiamata `choices` che riceve i pesi come argomento facoltativo.

In [ ]:
random.choices(words, weights=weights)

E riceve un altro argomento facoltativo, `k`, che specifica il numero di parole da selezionare.

In [ ]:
random_words = random.choices(words, weights=weights, k=6)
random_words

Il risultato è una lista di stringhe che possiamo unire in qualcosa che somiglia di più a una frase.

In [ ]:
' '.join(random_words)

Se scegli parole a caso dal libro, ti fai un'idea del vocabolario, ma una serie di parole casuali ha raramente senso perché non c'è alcuna relazione tra parole successive.
Per esempio, in una frase reale ci si aspetta che un articolo come "the" sia seguito da un aggettivo o da un nome, e probabilmente non da un verbo o da un avverbio.
Quindi il passo successivo è esaminare queste relazioni tra le parole.

## Bigrammi

Invece di guardare una parola alla volta, ora guarderemo sequenze di due parole, chiamate **bigrammi** (*bigrams*).
Una sequenza di tre parole si chiama **trigramma** (*trigram*), e una sequenza con un numero non specificato di parole si chiama **n-gramma** (*n-gram*).

Scriviamo un programma che trova tutti i bigrammi del libro e il numero di volte in cui compare ciascuno.
Per memorizzare i risultati, useremo un dizionario in cui

* Le chiavi sono tuple di stringhe che rappresentano bigrammi, e

* I valori sono interi che rappresentano frequenze.

Chiamiamolo `bigram_counter`.

In [ ]:
bigram_counter = {}

La funzione seguente riceve come parametro una lista di due stringhe.
Per prima cosa crea una tupla con le due stringhe, che si può usare come chiave di un dizionario.
Poi aggiunge la chiave a `bigram_counter`, se non esiste, oppure ne incrementa la frequenza, se esiste.

In [ ]:
def count_bigram(bigram):
    key = tuple(bigram)
    if key not in bigram_counter:
        bigram_counter[key] = 1
    else:
        bigram_counter[key] += 1

Mentre scorriamo il libro, dobbiamo tenere traccia di ogni coppia di parole consecutive.
Quindi, se vediamo la sequenza "man is not truly one", aggiungeremo i bigrammi "man is", "is not", "not truly", e così via.

Per tenere traccia di questi bigrammi, useremo una lista chiamata `window` (finestra), perché è come una finestra che scorre sulle pagine del libro, mostrando solo due parole alla volta.
All'inizio `window` è vuota.

In [ ]:
window = []

Useremo la funzione seguente per elaborare le parole una alla volta.

In [ ]:
def process_word(word):
    window.append(word)
    
    if len(window) == 2:
        count_bigram(window)
        window.pop(0)

La prima volta che questa funzione viene chiamata, aggiunge la parola data a `window`.
Poiché nella finestra c'è una sola parola, non abbiamo ancora un bigramma, quindi la funzione termina.

La seconda volta che viene chiamata – e ogni volta successiva – aggiunge una seconda parola a `window`.
Poiché nella finestra ci sono due parole, chiama `count_bigram` per tenere traccia di quante volte compare ogni bigramma.
Poi usa `pop` per rimuovere la prima parola dalla finestra.

Il programma seguente scorre le parole del libro e le elabora una alla volta.

In [ ]:
for line in open(filename, encoding='utf-8'):
    for word in split_line(line):
        word = clean_word(word)
        process_word(word)

Il risultato è un dizionario che fa corrispondere a ogni bigramma il numero di volte in cui compare.
Possiamo usare `print_most_common` per vedere i bigrammi più comuni.

In [ ]:
print_most_common(bigram_counter)

Guardando questi risultati, possiamo farci un'idea di quali coppie di parole è più probabile che compaiano insieme.
Possiamo anche usare i risultati per generare testo casuale, così.

In [ ]:
random.seed(0)

In [ ]:
bigrams = list(bigram_counter)
weights = bigram_counter.values()
random_bigrams = random.choices(bigrams, weights=weights, k=6)

`bigrams` è una lista dei bigrammi che compaiono nel libro.
`weights` è una lista delle loro frequenze, quindi `random_bigrams` è un campione in cui la probabilità che un bigramma venga selezionato è proporzionale alla sua frequenza.

Ecco i risultati.

In [ ]:
for pair in random_bigrams:
    print(' '.join(pair), end=' ')

Questo modo di generare testo è migliore della scelta di parole casuali, ma continua a non avere molto senso.

## Analisi di Markov

Possiamo fare di meglio con l'analisi del testo basata sulle catene di Markov, che calcola, per ogni parola di un testo, l'elenco delle parole che la seguono.
Come esempio, analizzeremo questo testo della canzone dei Monty Python *Eric, the Half a Bee*:

In [ ]:
song = """
Half a bee, philosophically,
Must, ipso facto, half not be.
But half the bee has got to be
Vis a vis, its entity. D'you see?
"""

Per memorizzare i risultati, useremo un dizionario che fa corrispondere a ogni parola la lista delle parole che la seguono.

In [ ]:
successor_map = {}

Come esempio, iniziamo con le prime due parole della canzone.

In [ ]:
first = 'half'
second = 'a'

Se la prima parola non è in `successor_map`, dobbiamo aggiungere un nuovo elemento che fa corrispondere alla prima parola una lista contenente la seconda.

In [ ]:
successor_map[first] = [second]
successor_map

Se la prima parola è già nel dizionario, possiamo cercarla per ottenere la lista dei successori visti finora e aggiungere quello nuovo.

In [ ]:
first = 'half'
second = 'not'

successor_map[first].append(second)
successor_map

La funzione seguente incapsula questi passi.

In [ ]:
def add_bigram(bigram):
    first, second = bigram
    
    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

Se lo stesso bigramma compare più di una volta, la seconda parola viene aggiunta alla lista più di una volta.
In questo modo, `successor_map` tiene traccia di quante volte compare ogni successore.

Come nella sezione precedente, useremo una lista chiamata `window` per memorizzare coppie di parole consecutive.
E useremo la funzione seguente per elaborare le parole una alla volta.

In [ ]:
def process_word_bigram(word):
    window.append(word)
    
    if len(window) == 2:
        add_bigram(window)
        window.pop(0)

Ecco come la usiamo per elaborare le parole della canzone.

In [ ]:
successor_map = {}
window = []

for word in song.split():
    word = clean_word(word)
    process_word_bigram(word)

Ed ecco i risultati.

In [ ]:
successor_map

La parola `'half'` può essere seguita da `'a'`, `'not'` o `'the'`.
La parola `'a'` può essere seguita da `'bee'` o `'vis'`.
La maggior parte delle altre parole compare una sola volta, quindi è seguita da una sola parola.

Ora analizziamo il libro.

In [ ]:
successor_map = {}
window = []

for line in open(filename, encoding='utf-8'):
    for word in split_line(line):
        word = clean_word(word)
        process_word_bigram(word)

Possiamo cercare qualsiasi parola e trovare le parole che possono seguirla.

In [ ]:
# I used this cell to find a predecessor with a good number of possible successors
# and at least one repeated word.

def has_duplicates(t):
    return len(set(t)) < len(t)

for key, value in successor_map.items():
    if len(value) == 7 and has_duplicates(value):
        print(key, value)

In [ ]:
successor_map['going']

In questa lista di successori, nota che la parola `'to'` compare tre volte – gli altri successori compaiono una sola volta.

## Generare testo

Possiamo usare i risultati della sezione precedente per generare nuovo testo con le stesse relazioni tra parole consecutive dell'originale.
Ecco come funziona:

* Partendo da una parola qualsiasi che compare nel testo, cerchiamo i suoi possibili successori e ne scegliamo uno a caso.

* Poi, usando la parola scelta, cerchiamo i suoi possibili successori e ne scegliamo uno a caso.

Possiamo ripetere questo procedimento per generare quante parole vogliamo.
Come esempio, iniziamo con la parola `'although'`.
Ecco le parole che possono seguirla.

In [ ]:
word = 'although'
successors = successor_map[word]
successors

In [ ]:
# this cell initializes the random number generator so it 
# starts at the same point in the sequence each time this
# notebook runs.

random.seed(2)

Possiamo usare `choice` per scegliere dalla lista con uguale probabilità.

In [ ]:
word = random.choice(successors)
word

Se la stessa parola compare più di una volta nella lista, è più probabile che venga selezionata.

Ripetendo questi passi, possiamo usare il ciclo seguente per generare una serie più lunga.

In [ ]:
for i in range(10):
    successors = successor_map[word]
    word = random.choice(successors)
    print(word, end=' ')

Il risultato somiglia di più a una frase vera, ma continua a non avere molto senso.

Possiamo fare di meglio usando più di una parola come chiave in `successor_map`.
Per esempio, possiamo creare un dizionario che fa corrispondere a ogni bigramma – o trigramma – la lista delle parole che seguono.
Come esercizio, avrai l'occasione di implementare questa analisi e vedere come appaiono i risultati.

## Debugging

A questo punto stiamo scrivendo programmi più consistenti, e potresti scoprire di passare più tempo a fare debugging.
Se sei bloccato su un bug difficile, ecco alcune cose da provare:

* Leggere: esamina il tuo codice, rileggilo a te stesso e verifica che dica ciò che intendevi dire.

* Eseguire: sperimenta facendo modifiche ed eseguendo versioni diverse. Spesso, se visualizzi la cosa giusta nel punto giusto del programma, il problema diventa evidente, ma a volte devi costruire un'impalcatura.

* Rimuginare: prenditi del tempo per pensare! Che tipo di errore è: di sintassi, di runtime
    o semantico? Quali informazioni puoi ricavare dai messaggi di errore
    o dall'output del programma? Che tipo di errore potrebbe causare
    il problema che vedi? Qual è l'ultima cosa che hai cambiato prima che
    comparisse il problema?

* Paperella di gomma (*rubberducking*): se spieghi il problema a qualcun altro, a volte trovi la
    risposta prima di aver finito di fare la domanda. Spesso non ti serve
    l'altra persona; potresti semplicemente parlare a una paperella di gomma. Ed è
    questa l'origine della nota strategia chiamata **debugging con la paperella di gomma**
    (*rubber duck debugging*). Non me lo sto inventando – vedi
    <https://en.wikipedia.org/wiki/Rubber_duck_debugging>.

* Ritirarsi: a un certo punto, la cosa migliore da fare è tornare indietro – annullando le modifiche
    recenti – finché non arrivi a un programma che funziona. Poi puoi iniziare a ricostruire.

* Riposare: se dai una pausa al tuo cervello, a volte troverà il problema per te.

I programmatori alle prime armi a volte si bloccano su una di queste attività e dimenticano le altre. Ogni attività ha il suo modo di fallire.

Per esempio, leggere il codice funziona se il problema è un errore di battitura, ma non se il problema è un'incomprensione concettuale.
Se non capisci cosa fa il tuo programma, puoi leggerlo 100 volte e non vedere mai l'errore, perché l'errore è nella tua testa.

Fare esperimenti può funzionare, soprattutto se esegui test piccoli e semplici.
Ma se fai esperimenti senza pensare o senza leggere il tuo codice, può volerci molto tempo per capire cosa sta succedendo.

Devi prenderti il tempo per pensare. Il debugging è come una scienza sperimentale. Dovresti avere almeno un'ipotesi su quale sia il problema. Se ci sono due o più possibilità, cerca di pensare a un test che ne escluda una.

Ma anche le migliori tecniche di debugging falliscono se ci sono troppi
errori, o se il codice che cerchi di correggere è troppo grande e complicato.
A volte l'opzione migliore è ritirarsi, semplificando il programma finché
non torni a qualcosa che funziona.

I programmatori alle prime armi sono spesso riluttanti a ritirarsi perché non
sopportano di cancellare una riga di codice (anche se è sbagliata). Se ti fa
sentire meglio, copia il tuo programma in un altro file prima di iniziare
a sfoltirlo. Poi puoi ricopiare i pezzi uno alla volta.

Trovare un bug difficile richiede di leggere, eseguire, rimuginare, ritirarsi e a volte riposare.
Se ti blocchi su una di queste attività, prova le altre.

## Glossario

**valore predefinito** (*default value*):
Il valore assegnato a un parametro se non viene fornito alcun argomento.

**sovrascrivere** (*override*):
Sostituire un valore predefinito con un argomento.

**deterministico** (*deterministic*):
Un programma deterministico fa la stessa cosa ogni volta che viene eseguito, a parità di input.

**pseudocasuale** (*pseudorandom*):
Una sequenza di numeri pseudocasuale sembra casuale, ma è generata da un programma deterministico.

**bigramma** (*bigram*):
Una sequenza di due elementi, spesso parole.

**trigramma** (*trigram*):
Una sequenza di tre elementi.

**n-gramma** (*n-gram*):
Una sequenza di un numero non specificato di elementi.

**debugging con la paperella di gomma** (*rubber duck debugging*):
Un modo di fare debugging spiegando un problema ad alta voce a un oggetto inanimato.

## Esercizi

In [ ]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

### Chiedi a un assistente virtuale

In `add_bigram`, l'istruzione `if` crea una nuova lista o aggiunge un elemento a una lista esistente, a seconda che la chiave sia già nel dizionario.

In [ ]:
def add_bigram(bigram):
    first, second = bigram
    
    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

I dizionari forniscono un metodo chiamato `setdefault` che possiamo usare per fare la stessa cosa in modo più conciso.
Chiedi a un assistente virtuale come funziona, oppure copia `add_bigram` in un assistente virtuale e chiedi "Puoi riscriverla usando `setdefault`?"

In questo capitolo abbiamo implementato l'analisi e la generazione di testo con le catene di Markov.
Se sei curioso, puoi chiedere a un assistente virtuale maggiori informazioni sull'argomento.
Una delle cose che potresti scoprire è che gli assistenti virtuali usano algoritmi simili sotto molti aspetti – ma anche diversi in modi importanti.
Chiedi a un assistente "Quali sono le differenze tra i modelli linguistici di grandi dimensioni come GPT e l'analisi del testo con le catene di Markov?"

### Esercizio

Scrivi una funzione che conta il numero di volte in cui compare ogni trigramma (sequenza di tre parole).
Se testi la tua funzione con il testo di _Dr. Jekyll and Mr. Hyde_, dovresti scoprire che il trigramma più comune è "said the lawyer".

Suggerimento: scrivi una funzione chiamata `count_trigram` simile a `count_bigram`. Poi scrivi una funzione chiamata `process_word_trigram` simile a `process_word_bigram`.

In [ ]:
# Solution goes here

In [ ]:
# Solution goes here

Puoi usare il ciclo seguente per leggere il libro ed elaborare le parole.

In [ ]:
trigram_counter = {}
window = []

for line in open(filename, encoding='utf-8'):
    for word in split_line(line):
        word = clean_word(word)
        process_word_trigram(word)

Poi usa `print_most_common` per trovare i trigrammi più comuni del libro.

In [ ]:
print_most_common(trigram_counter)

### Esercizio

Ora implementiamo l'analisi del testo con le catene di Markov usando una mappatura da ogni bigramma a una lista di possibili successori.

Partendo da `add_bigram`, scrivi una funzione chiamata `add_trigram` che riceve una lista di tre parole e aggiunge o aggiorna un elemento in `successor_map`, usando le prime due parole come chiave e la terza come possibile successore.

In [ ]:
# Solution goes here

Ecco una versione di `process_word_trigram` che chiama `add_trigram`.

In [ ]:
def process_word_trigram(word):
    window.append(word)
    
    if len(window) == 3:
        add_trigram(window)
        window.pop(0)

Puoi usare il ciclo seguente per testare la tua funzione con il testo di "Eric, the Half a Bee".

In [ ]:
successor_map = {}
window = []

for string in song.split():
    word = string.strip(punctuation).lower()
    process_word_trigram(word)

Se la tua funzione funziona come previsto, il predecessore `('half', 'a')` dovrebbe corrispondere a una lista con il solo elemento `'bee'`.
In effetti, guarda caso, ogni bigramma di questa canzone compare una sola volta, quindi tutti i valori in `successor_map` hanno un solo elemento.

In [ ]:
successor_map

Puoi usare il ciclo seguente per testare la tua funzione con le parole del libro.

In [ ]:
successor_map = {}
window = []

for line in open(filename, encoding='utf-8'):
    for word in split_line(line):
        word = clean_word(word)
        process_word_trigram(word)

Nel prossimo esercizio userai i risultati per generare nuovo testo casuale.

### Esercizio

Per questo esercizio assumeremo che `successor_map` sia un dizionario che fa corrispondere a ogni bigramma la lista delle parole che lo seguono.

In [ ]:
# this cell initializes the random number generator so it 
# starts at the same point in the sequence each time this
# notebook runs.

random.seed(3)

Per generare testo casuale, inizieremo scegliendo una chiave a caso da `successor_map`.

In [ ]:
successors = list(successor_map)
bigram = random.choice(successors)
bigram

Ora scrivi un ciclo che genera altre 50 parole seguendo questi passi:

1. In `successor_map`, cerca la lista delle parole che possono seguire `bigram`.

2. Scegline una a caso e stampala.

3. Per l'iterazione successiva, crea un nuovo bigramma che contiene la seconda parola di `bigram` e il successore scelto.

Per esempio, se partiamo dal bigramma `('doubted', 'if')` e scegliamo `'from'` come successore, il bigramma successivo è `('if', 'from')`.

In [ ]:
# Solution goes here

Se tutto funziona, dovresti scoprire che il testo generato ha uno stile riconoscibilmente simile all'originale, e che alcune frasi hanno senso, ma il testo potrebbe saltare da un argomento all'altro.

Come esercizio bonus, modifica la tua soluzione degli ultimi due esercizi per usare trigrammi come chiavi in `successor_map`, e osserva che effetto ha sui risultati.

[Pensare in Python](https://pensare-in-python.github.io/3a-edizione/) – traduzione italiana non ufficiale di [Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Licenza del codice: [MIT License](https://mit-license.org/)

Licenza del testo (originale e traduzione): [Creative Commons Attribuzione - Non commerciale - Condividi allo stesso modo 4.0 Internazionale](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.it)